In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import  Path
path = r"H:\202304_202407_guage\pre_hor\pre_hor_20230601000000.csv"
# path = r"H:\202304_202407_guage\pre_hor\pre_hor_20230801000000.csv"
df = pd.read_csv(path,header=None, low_memory=False)
df_new=pd.DataFrame()
df_new['station']=df[0]  
df_new['province']=df[1]
df_new['cnty']=df[4]
df_new["lon"] = df[11]
df_new["lat"] = df[10]
df_new["pre"] = df[14]
df_new["city"] = df[3]
df_new = df_new.drop(index=[0])  # 删第一行
df_new["lon"] = df_new["lon"].map(float)
df_new["lat"] = df_new["lat"].map(float)
df_new["pre"] = df_new["pre"].map(float)

df_clear = df_new.drop(
    df_new[#(df_new["pre"] > 888888) | 
        (df_new["lon"] > 122) |
        (df_new["lon"] < 111) |
        (df_new["lat"] > 37) |
        (df_new["lat"] < 30) 
    ].index
)
df_reset = df_clear.reset_index(drop=True)

In [ ]:
path = r"H:\202304_202407_guage\pre_hor\pre_hor_20230601000000.csv"
# path = r"H:\202304_202407_guage\pre_hor\pre_hor_20230801000000.csv"
df = pd.read_csv(path,header=None, low_memory=False)

In [ ]:
df

In [ ]:
df_reset

In [ ]:
#筛选没有漏测的地基站点

dirpath_data = Path(r"H:\202304_202407_guage\pre_hor")
path = r"H:\202304_202407_guage\pre_hor"
files = os.listdir(path)

all_lon_lat=set(list(zip(df_reset['lon'], df_reset['lat'])))

i = 0  # i为file计数
for file in files:
    print(file)
    i += 1
    filepath = dirpath_data / file
    if int(file[8:18])<2023050100:
        continue
    if int(file[8:18])>=2024050100:
        break
    df = pd.read_csv(filepath,  header=None, low_memory=False, usecols=[10, 11, 14], skiprows=[0],dtype=float)
    lat, lon ,pre= df[10].values, df[11].values,df[14].values
    lat=lat[pre<888888]
    lon=lon[pre<888888]
    lon_lat = list(zip(lon, lat))
    lon_lat_set = set(lon_lat)

    all_lon_lat=all_lon_lat.intersection(lon_lat_set)


new_lons = [coord[0] for coord in all_lon_lat]
new_lats = [coord[1] for coord in all_lon_lat]


In [ ]:
new_lons

In [ ]:
fenbian=1#0.1
p=int(7/fenbian)
q=int(11/fenbian)
num=np.zeros((p,q))
for i in range(len(new_lons)):
    x=int((new_lons[i]-111)/fenbian)
    y=int((new_lats[i]-30)/fenbian)
    num[y,x]=num[y,x]+1

In [ ]:
df_num = pd.DataFrame(columns=['indexi', 'indexj', 'num'])
for i in range(len(new_lons)):
    x=int((new_lons[i]-111)/fenbian)
    y=int((new_lats[i]-30)/fenbian)
    condition = (df_num['indexi'] == y) & (df_num['indexj'] == x)
    
    if df_num[condition].empty:
        # 如果记录不存在，添加新的记录
        new_row = pd.DataFrame({'indexi': [y], 'indexj': [x], 'num': [1]})
        df_num = pd.concat([df_num, new_row], ignore_index=True)
    else:
        # 如果记录存在，更新计数
        df_num.loc[condition, 'num'] += 1

In [ ]:
new_lats=np.array(new_lats)
new_lons=np.array(new_lons)
df_lonlats =pd.DataFrame({'indexi':((new_lats-30)/fenbian).astype(int),
					'indexj':((new_lons-111)/fenbian).astype(int),
                    'lons':new_lons,'lats':new_lats,
					})

In [ ]:
df_posi = pd.merge(df_lonlats, df_num, on=['indexi', 'indexj'], how='outer')

In [ ]:
# df_posi.to_csv('df_posi_1.csv',index=False)
df_posi=pd.read_csv('df_posi_01.csv')
df_posi

In [ ]:
df_reset

In [ ]:
df_staname=pd.merge(df_reset,df_posi,how='inner',left_on=['lon','lat'],right_on=['lons','lats'])

In [ ]:
df_staname[ (df_staname['city']=='黄山市')]

In [ ]:
df_staname[(df_staname['cnty']=='安泽县')]